# Phase 2 — Measure
## 04 — SOA Transformation

### Objective

Transform the Master SOA source into a governed analytical dataset linked to the canonical Product Master.

### Business Definition

SOA = Sell Out Allowance.

It represents an additional supplier allowance/discount applicable to a product during a defined promotional date window.

### Phase 2 Goals

- Validate the SOA source structure.
- Standardize Model identifiers.
- Map every SOA record to the canonical Product Master.
- Validate allowance values.
- Validate promotional date windows.
- Investigate overlapping SOA periods.
- Preserve the original supplier information.
- Build the governed `fact_soa` dataset.

### Important Rule

Phase 2 validates and governs SOA data.

It does not yet apply SOA to selling-price calculations or dynamic-pricing decisions.

In [1]:
## Imports and paths
from pathlib import Path
import pandas as pd
import numpy as np

current_path = Path.cwd()

if current_path.name == "Phase_2_Measure":
    PROJECT_ROOT = current_path.parent.parent
elif current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root :", PROJECT_ROOT)
print("Raw data dir :", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)

Project root : d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System
Raw data dir : d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\raw
Processed dir: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed


In [2]:
## Locate SOA source

soa_files = list(
    RAW_DIR.glob("*SOA*.xlsx")
)

print("SOA files found:", len(soa_files))

for file in soa_files:
    print(" -", file.name)

if len(soa_files) != 1:
    raise ValueError(
        f"Expected exactly 1 SOA workbook, found {len(soa_files)}"
    )

SOA_FILE = soa_files[0]

print("\nSelected SOA file:", SOA_FILE.name)

SOA files found: 1
 - Master-SOA.xlsx

Selected SOA file: Master-SOA.xlsx


In [3]:
soa_excel = pd.ExcelFile(SOA_FILE)

print("Workbook:", SOA_FILE.name)
print("Sheets  :", soa_excel.sheet_names)

Workbook: Master-SOA.xlsx
Sheets  : ['Master_SOA']


In [4]:
## Load SOA
soa_raw = pd.read_excel(
    SOA_FILE,
    sheet_name=soa_excel.sheet_names[0]
)

print("SOA shape:", soa_raw.shape)

display(soa_raw.head(10))

SOA shape: (412, 5)


,Model,Description,Starts,Ends,SOA
0,DW60A8050FB/EU,Samsung Black St/St 14 Place,2026-02-18,2026-02-28,216.72
1,WF-7830DTWF,Epson C11CH68401 Workforce,2026-02-16,2026-02-28,5.59
2,WF-2930DWF,Epson WF-2930DWF WorkForce,2026-02-16,2026-02-28,2.80
3,XP4200,Epson XP-4200 Printer,2026-02-16,2026-02-28,5.59
4,XP3200,Epson Expression Home,2026-02-16,2026-02-28,2.80
5,XP2200,Epson XP-2200 Printer,2026-02-16,2026-02-28,2.80
6,ET-2950,Epson EcoTank ET-2950,2026-02-16,2026-02-28,16.78
7,ET-2860,ET-2860 MULTIFUNCTION,2026-02-16,2026-02-28,16.78
8,JBLT670NCWHT,JBL White Tune 670 Headphones,2026-02-13,2026-02-28,10.14
9,JBLT670NCBLK,"JBL Tune 670NC, On-ear",2026-02-13,2026-02-28,10.14


In [5]:
### Step 1 — Schema Profiling & Data types
print("SOA columns:")

for i, col in enumerate(soa_raw.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nTotal columns:", len(soa_raw.columns))

SOA columns:
01. Model
02. Description
03. Starts
04. Ends
05. SOA

Total columns: 5


In [6]:
soa_schema = pd.DataFrame({
    "Column": soa_raw.columns,
    "Data_Type": [
        str(soa_raw[col].dtype)
        for col in soa_raw.columns
    ],
    "Missing": [
        soa_raw[col].isna().sum()
        for col in soa_raw.columns
    ],
    "Unique": [
        soa_raw[col].nunique(dropna=True)
        for col in soa_raw.columns
    ]
})

display(soa_schema)

,Column,Data_Type,Missing,Unique
0,Model,object,0,412
1,Description,object,0,377
2,Starts,datetime64[ns],0,22
3,Ends,datetime64[ns],0,12
4,SOA,float64,0,228


In [7]:
## Basic record integrity
print("Total rows:", len(soa_raw))

print(
    "Unique Model:",
    soa_raw["Model"].nunique(dropna=True)
)

print(
    "Missing Model:",
    soa_raw["Model"].isna().sum()
)

print(
    "Duplicate Model:",
    soa_raw["Model"].duplicated().sum()
)

print(
    "Full-row duplicates:",
    soa_raw.duplicated().sum()
)

Total rows: 412
Unique Model: 412
Missing Model: 0
Duplicate Model: 0
Full-row duplicates: 0


In [8]:
## Missing-value profile
missing_profile = (
    soa_raw
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

missing_profile["Missing_%"] = (
    missing_profile["Missing_Count"]
    / len(soa_raw)
    * 100
).round(2)

display(missing_profile)

,Missing_Count,Missing_%
Model,0,0.0
Description,0,0.0
Starts,0,0.0
Ends,0,0.0
SOA,0,0.0


In [9]:
display(
    soa_raw.head(20)
)

,Model,Description,Starts,Ends,SOA
0,DW60A8050FB/EU,Samsung Black St/St 14 Place,2026-02-18,2026-02-28,216.72
1,WF-7830DTWF,Epson C11CH68401 Workforce,2026-02-16,2026-02-28,5.59
2,WF-2930DWF,Epson WF-2930DWF WorkForce,2026-02-16,2026-02-28,2.80
3,XP4200,Epson XP-4200 Printer,2026-02-16,2026-02-28,5.59
4,XP3200,Epson Expression Home,2026-02-16,2026-02-28,2.80
5,XP2200,Epson XP-2200 Printer,2026-02-16,2026-02-28,2.80
6,ET-2950,Epson EcoTank ET-2950,2026-02-16,2026-02-28,16.78
7,ET-2860,ET-2860 MULTIFUNCTION,2026-02-16,2026-02-28,16.78
8,JBLT670NCWHT,JBL White Tune 670 Headphones,2026-02-13,2026-02-28,10.14
9,JBLT670NCBLK,"JBL Tune 670NC, On-ear",2026-02-13,2026-02-28,10.14


### Step 1 — SOA Business Field Validation

In [20]:
soa_amount_profile = pd.Series({
    "Records": len(soa_raw),
    "Missing_SOA": soa_raw["SOA"].isna().sum(),
    "Zero_SOA": (soa_raw["SOA"] == 0).sum(),
    "Negative_SOA": (soa_raw["SOA"] < 0).sum(),
    "Positive_SOA": (soa_raw["SOA"] > 0).sum(),
    "Minimum_SOA": soa_raw["SOA"].min(),
    "Maximum_SOA": soa_raw["SOA"].max(),
    "Mean_SOA": soa_raw["SOA"].mean(),
    "Median_SOA": soa_raw["SOA"].median()
}).to_frame("Value")

display(soa_amount_profile)

,Value
Records,412.000000
Missing_SOA,0.000000
Zero_SOA,0.000000
Negative_SOA,0.000000
Positive_SOA,412.000000
Minimum_SOA,0.430000
Maximum_SOA,918.480000
Mean_SOA,72.099199
Median_SOA,29.750000


In [21]:
## Highest and lowest SOA values
print("Top 15 highest SOA values:")

display(
    soa_raw[
        ["Model", "Description", "Starts", "Ends", "SOA"]
    ]
    .sort_values("SOA", ascending=False)
    .head(15)
)

print("\nTop 15 lowest SOA values:")

display(
    soa_raw[
        ["Model", "Description", "Starts", "Ends", "SOA"]
    ]
    .sort_values("SOA", ascending=True)
    .head(15)
)

Top 15 highest SOA values:


,Model,Description,Starts,Ends,SOA
197,OLED83C44LA.AE,"LG 83"" OLED Television",2025-12-03,2026-01-27,918.48
214,OLED77B42LA.AE,"LG 77"" OLED B4 Smart TV",2025-12-03,2026-01-27,866.88
23,RF65DG9H0ESR/E,Samsung Family Hub USA Fridge,2026-02-11,2026-02-28,638.98
24,RF65DG9H0EB1/E,Samsung Black Family Hub USA,2026-02-11,2026-02-28,638.98
209,OLED65B42LA.AE,"LG 65"" OLED Television",2025-12-03,2026-01-27,624.36
207,OLED83G45LW.AE,"LG OLED 83"" G45 TV",2025-12-03,2026-01-27,614.04
210,OLED65B46LA.AE,"LG 65"" OLED TV",2025-12-03,2026-01-27,516.00
218,75QNED80T6A.AE,"LG 75"" QNED80 AI 4K Smart TV",2025-12-03,2026-01-27,485.04
204,OLED55B42LA.AE,"LG 55"" OLED Smart TV",2025-12-03,2026-01-27,459.24
198,75QNED87T6B.AE,"LG 75"" QNED 4K Smart",2025-12-03,2026-01-27,426.56



Top 15 lowest SOA values:


,Model,Description,Starts,Ends,SOA
38,DECOM4KIT,DECOM4 3pk Mesh Home Wi-Fi,2026-02-04,2026-02-28,0.43
39,DECOX50OUTDOO,TP Link Mesh Wifi System Dual,2026-02-04,2026-02-28,0.43
37,TAPOT110,Tapo Smart Contact Sensor,2026-02-04,2026-02-28,0.43
40,MR200,TPLINK WIRELESS 4G ROUTER,2026-02-04,2026-02-28,0.43
58,TAPOL530E,Tapo Smart WiFi Multicolur Screw,2026-02-04,2026-02-28,0.43
61,TAPOL9205,"Tapo Smart Light Strip,",2026-02-04,2026-02-28,0.43
48,TAPOC100,Tapo Home Security Wifi Camera,2026-02-04,2026-02-28,0.43
43,TAPOL510E,Tapo Smart Wifi Dimmable,2026-02-04,2026-02-28,0.43
51,TAPOC410,Smart Wire-Free Indoor/Outdoor,2026-02-04,2026-02-28,0.43
52,TAPOC500,Tapo Outdoor Pan/Tilt Security,2026-02-04,2026-02-28,0.43


### Step 2 — Promotional Date Window Validation

In [10]:
## Date range profile
date_profile = pd.Series({
    "Earliest_Start": soa_raw["Starts"].min(),
    "Latest_Start": soa_raw["Starts"].max(),
    "Earliest_End": soa_raw["Ends"].min(),
    "Latest_End": soa_raw["Ends"].max(),
    "Unique_Start_Dates": soa_raw["Starts"].nunique(),
    "Unique_End_Dates": soa_raw["Ends"].nunique()
}).to_frame("Value")

display(date_profile)

,Value
Earliest_Start,2025-11-26 00:00:00
Latest_Start,2026-02-18 00:00:00
Earliest_End,2025-03-04 00:00:00
Latest_End,2026-03-17 00:00:00
Unique_Start_Dates,22
Unique_End_Dates,12


In [11]:
## Check Invalid date windows
invalid_windows = soa_raw[
    soa_raw["Starts"] > soa_raw["Ends"]
].copy()

print(
    "Invalid SOA date windows:",
    len(invalid_windows)
)

display(invalid_windows)

Invalid SOA date windows: 1


,Model,Description,Starts,Ends,SOA
304,SM-X210NZAAEUB,Galaxy Tab A9+ GRAY WIFI,2026-02-12,2025-03-04,34.87


In [12]:
## Calculate allowance-window duration
soa_check = soa_raw.copy()

soa_check["Window_Days"] = (
    soa_check["Ends"] -
    soa_check["Starts"]
).dt.days + 1

display(
    soa_check[
        ["Model", "Description", "Starts", "Ends", "Window_Days", "SOA"]
    ]
    .sort_values("Window_Days")
    .head(20)
)

,Model,Description,Starts,Ends,Window_Days,SOA
304,SM-X210NZAAEUB,Galaxy Tab A9+ GRAY WIFI,2026-02-12,2025-03-04,-344,34.87
124,AF180UK,Ninja Air Fryer MAX PRO 6.2L,2026-01-23,2026-01-27,5,14.17
303,WW10FG5U34AEE,Samsung White 10kg 1400 Spin,2026-02-12,2026-02-17,6,43.50
312,TB201UK,Ninja Detect Power Blender Pro,2026-02-11,2026-02-17,7,25.83
316,IW4621UKT,Shark Vacuum Cleaner,2026-02-11,2026-02-17,7,98.75
319,DZ300UK,Ninja 6-in-1 Dual Zone Air Fryer,2026-02-11,2026-02-17,7,31.67
314,S1000UK,Shark Pro Steam Mop,2026-02-11,2026-02-17,7,14.17
313,S8201UK,Shark Steam & Scrub Automatic,2026-02-11,2026-02-17,7,14.17
311,AF160UK,Ninja Air Fryer Max,2026-02-11,2026-02-17,7,21.25
310,AF100UK,Ninja Air Fryer,2026-02-11,2026-02-17,7,14.17


In [13]:
## Duration profile
window_profile = pd.Series({
    "Minimum_Window_Days": soa_check["Window_Days"].min(),
    "Maximum_Window_Days": soa_check["Window_Days"].max(),
    "Mean_Window_Days": soa_check["Window_Days"].mean(),
    "Median_Window_Days": soa_check["Window_Days"].median(),
    "Unique_Window_Lengths": soa_check["Window_Days"].nunique()
}).to_frame("Value")

display(window_profile)

,Value
Minimum_Window_Days,-344.000000
Maximum_Window_Days,63.000000
Mean_Window_Days,23.082524
Median_Window_Days,25.000000
Unique_Window_Lengths,25.000000


In [14]:
## Distribution of SOA window lengths
window_distribution = (
    soa_check["Window_Days"]
    .value_counts()
    .sort_index()
    .rename_axis("Window_Days")
    .to_frame("Records")
)

display(window_distribution)

,Records
Window_Days,
-344,1
5,1
6,1
7,74
8,4
9,27
11,1
12,27
13,7


### Step 3 — Model Identity Validation

In [15]:
## Normalize SOA model
soa_check["Product_Key_Normalized"] = (
    soa_check["Model"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print(
    "Unique raw Models       :",
    soa_check["Model"].nunique()
)

print(
    "Unique normalized Models:",
    soa_check["Product_Key_Normalized"].nunique()
)

print(
    "Normalization collisions:",
    len(soa_check) -
    soa_check["Product_Key_Normalized"].nunique()
)

Unique raw Models       : 412
Unique normalized Models: 412
Normalization collisions: 0


In [16]:
## Check Product Master availability
PRODUCT_MASTER_FILE = PROCESSED_DIR / "dim_product.csv"

if not PRODUCT_MASTER_FILE.exists():
    raise FileNotFoundError(
        f"Product Master not found: {PRODUCT_MASTER_FILE}"
    )

dim_product = pd.read_csv(
    PRODUCT_MASTER_FILE,
    dtype={"Product_Key": "string"}
)

print("Product Master rows:", len(dim_product))
print("Unique Product_ID  :", dim_product["Product_ID"].nunique())
print("Unique Product_Key :", dim_product["Product_Key"].nunique())

display(dim_product.head())

Product Master rows: 1436
Unique Product_ID  : 1436
Unique Product_Key : 1436


,Product_ID,Product_Key,Product_Description,Product_Category,Record_Type,Source_Status,Source_Count,In_Sales,In_Stock,In_SOA,Sales_Stock_Code,Sales_Description,Sales_Category,Stock_Model,Stock_Description,Stock_Category,SOA_Model,SOA_Description
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,NaN,NaN,NaN,NaN,NaN
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,NaN,NaN,NaN,NaN,NaN
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,NaN,NaN,NaN,NaN,NaN
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,NaN,NaN,NaN,NaN,NaN
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,PRODUCT,SALES_ONLY,1,True,False,False,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,NaN,NaN,NaN,NaN,NaN


In [17]:
## Map SOA → Product Master
soa_mapped = soa_check.merge(
    dim_product[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Record_Type"
        ]
    ],
    left_on="Product_Key_Normalized",
    right_on="Product_Key",
    how="left",
    validate="one_to_one"
)

print("SOA source rows :", len(soa_check))
print("SOA mapped rows :", soa_mapped["Product_ID"].notna().sum())
print("Unmapped rows   :", soa_mapped["Product_ID"].isna().sum())

SOA source rows : 412
SOA mapped rows : 412
Unmapped rows   : 0


In [18]:
## Description comparison for audit only
soa_mapped["Description_Exact_Match"] = (
    soa_mapped["Description"]
    .astype(str)
    .str.strip()
    .str.upper()
    ==
    soa_mapped["Product_Description"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print(
    "Exact description matches:",
    soa_mapped["Description_Exact_Match"].sum()
)

print(
    "Different descriptions:",
    (~soa_mapped["Description_Exact_Match"]).sum()
)

Exact description matches: 390
Different descriptions: 22


In [19]:
display(
    soa_mapped.loc[
        ~soa_mapped["Description_Exact_Match"],
        [
            "Model",
            "Description",
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category"
        ]
    ].head(30)
)

,Model,Description,Product_ID,Product_Key,Product_Description,Product_Category
4,XP3200,Epson Expression Home,1422,XP3200,Epson Expression Home XP-3200,PRINTERS
20,RS90F66BEFEU,Samsung Black Family Hob USA,1045,RS90F66BEFEU,Samsung Black Family Hob USA FF,USA F/F
41,TAPOL510B,Tapo Smart Wifi Dimmable,1191,TAPOL510B,Tapo Smart Wifi Dimmable Bayonet,BULBS
61,TAPOL9205,"Tapo Smart Light Strip,",1195,TAPOL9205,"Tapo Smart Light Strip, Multicolour",BULBS
81,DV90DG6845LBU1,Samsung Series 6 9kg Heat Pump,398,DV90DG6845LBU1,Samsung Series 6 9kg Heat Pump Dryer,TUMBLE DRYERS
83,DV90DG6845LEU1,Sumsung White 9kg Heat Pump,399,DV90DG6845LEU1,Sumsung White 9kg Heat Pump Dryer,TUMBLE DRYERS
85,DV90DB8845GBU1,Samsung Series 8 9kg Heat Pump,394,DV90DB8845GBU1,Samsung Series 8 9kg Heat Pump Dryer,TUMBLE DRYERS
92,700365,Sennheiser Momentum Tue,208,700365,Sennheiser Momentum Tue Wireless,HEADPHONES
106,23211,Luna Moonlight Grey Quiet Boil,57,23211,Luna Moonlight Grey Quiet Boil Kettle,KETTLES
107,23221,Luna Moonlight Grey 2 Slice,58,23221,Luna Moonlight Grey 2 Slice Toaster,TOASTERS


In [22]:
## Apply controlled correction for invalid data for 'SM-X210NZAAEUB' model
soa_clean = soa_raw.copy()

# Preserve original end date for audit
soa_clean["Original_Ends"] = soa_clean["Ends"]

# Add correction flag
soa_clean["Date_Correction_Flag"] = False

# Correct confirmed bad record
mask = (
    soa_clean["Model"] == "SM-X210NZAAEUB"
)

soa_clean.loc[
    mask,
    "Ends"
] = pd.Timestamp("2026-03-04")

soa_clean.loc[
    mask,
    "Date_Correction_Flag"
] = True

display(
    soa_clean.loc[
        mask,
        [
            "Model",
            "Description",
            "Starts",
            "Original_Ends",
            "Ends",
            "Date_Correction_Flag",
            "SOA"
        ]
    ]
)

,Model,Description,Starts,Original_Ends,Ends,Date_Correction_Flag,SOA
304,SM-X210NZAAEUB,Galaxy Tab A9+ GRAY WIFI,2026-02-12,2025-03-04,2026-03-04,True,34.87


In [23]:
## Revalidate all date windows
invalid_windows_after = soa_clean[
    soa_clean["Starts"] > soa_clean["Ends"]
]

print(
    "Invalid SOA date windows after correction:",
    len(invalid_windows_after)
)

display(invalid_windows_after)

Invalid SOA date windows after correction: 0


,Model,Description,Starts,Ends,SOA,Original_Ends,Date_Correction_Flag


In [26]:
## Recalculate SOA window duration
soa_clean["Window_Days"] = (
    soa_clean["Ends"]
    - soa_clean["Starts"]
).dt.days + 1

display(
    soa_clean.loc[
        mask,
        [
            "Model",
            "Starts",
            "Ends",
            "Window_Days",
            "SOA"
        ]
    ]
)

,Model,Starts,Ends,Window_Days,SOA
304,SM-X210NZAAEUB,2026-02-12,2026-03-04,21,34.87


In [27]:
## Recalculate window profile
window_profile_corrected = pd.Series({
    "Minimum_Window_Days": soa_clean["Window_Days"].min(),
    "Maximum_Window_Days": soa_clean["Window_Days"].max(),
    "Mean_Window_Days": soa_clean["Window_Days"].mean(),
    "Median_Window_Days": soa_clean["Window_Days"].median(),
    "Unique_Window_Lengths": soa_clean["Window_Days"].nunique()
}).to_frame("Value")

display(window_profile_corrected)

,Value
Minimum_Window_Days,5.000000
Maximum_Window_Days,63.000000
Mean_Window_Days,23.968447
Median_Window_Days,25.000000
Unique_Window_Lengths,24.000000


In [28]:
## Correction summary
print(
    "Corrected records:",
    soa_clean["Date_Correction_Flag"].sum()
)

print(
    "Invalid windows remaining:",
    (soa_clean["Starts"] > soa_clean["Ends"]).sum()
)

Corrected records: 1
Invalid windows remaining: 0


In [33]:

# ============================================================
# CREATE SOA + PRODUCT_ID WORKING TABLE
# ============================================================

soa_work = soa_clean.merge(
    dim_product[
        ["Product_ID", "Product_Key", "Product_Description", "Product_Category"]
    ],
    left_on="Model",
    right_on="Product_Key",
    how="left",
    validate="many_to_one"
)

print("SOA rows             :", len(soa_work))
print("Mapped Product_ID    :", soa_work["Product_ID"].notna().sum())
print("Unmapped Product_ID  :", soa_work["Product_ID"].isna().sum())
print("Unique products      :", soa_work["Product_ID"].nunique())

display(
    soa_work[
        [
            "Product_ID",
            "Model",
            "Description",
            "Product_Category",
            "Starts",
            "Ends",
            "Window_Days",
            "SOA"
        ]
    ].head(10)
)

SOA rows             : 412
Mapped Product_ID    : 412
Unmapped Product_ID  : 0
Unique products      : 412


,Product_ID,Model,Description,Product_Category,Starts,Ends,Window_Days,SOA
0,410,DW60A8050FB/EU,Samsung Black St/St 14 Place,NaN,2026-02-18,2026-02-28,11,216.72
1,1322,WF-7830DTWF,Epson C11CH68401 Workforce,NaN,2026-02-16,2026-02-28,13,5.59
2,1321,WF-2930DWF,Epson WF-2930DWF WorkForce,NaN,2026-02-16,2026-02-28,13,2.80
3,1423,XP4200,Epson XP-4200 Printer,NaN,2026-02-16,2026-02-28,13,5.59
4,1422,XP3200,Epson Expression Home,PRINTERS,2026-02-16,2026-02-28,13,2.80
5,1421,XP2200,Epson XP-2200 Printer,NaN,2026-02-16,2026-02-28,13,2.80
6,472,ET-2950,Epson EcoTank ET-2950,NaN,2026-02-16,2026-02-28,13,16.78
7,471,ET-2860,ET-2860 MULTIFUNCTION,NaN,2026-02-16,2026-02-28,13,16.78
8,653,JBLT670NCWHT,JBL White Tune 670 Headphones,NaN,2026-02-13,2026-02-28,16,10.14
9,651,JBLT670NCBLK,"JBL Tune 670NC, On-ear",NaN,2026-02-13,2026-02-28,16,10.14


In [34]:
## Validate intended SOA grain


soa_grain_cols = [
    "Product_ID",
    "Starts",
    "Ends"
]

grain_duplicate_mask = soa_work.duplicated(
    subset=soa_grain_cols,
    keep=False
)

grain_duplicates = (
    soa_work.loc[
        grain_duplicate_mask,
        [
            "Product_ID",
            "Model",
            "Description",
            "Starts",
            "Ends",
            "SOA"
        ]
    ]
    .sort_values(["Product_ID", "Starts", "Ends"])
)

print("SOA rows:", len(soa_work))
print(
    "Unique Product_ID + Starts + Ends:",
    soa_work[soa_grain_cols].drop_duplicates().shape[0]
)
print(
    "Rows participating in duplicate SOA grain:",
    grain_duplicate_mask.sum()
)

display(grain_duplicates)

SOA rows: 412
Unique Product_ID + Starts + Ends: 412
Rows participating in duplicate SOA grain: 0


,Product_ID,Model,Description,Starts,Ends,SOA


In [35]:
# ============================================================
# SOA PERIODS PER PRODUCT
# ============================================================

soa_periods_per_product = (
    soa_work
    .groupby(
        ["Product_ID", "Model"],
        as_index=False
    )
    .agg(
        SOA_Periods=("SOA", "size"),
        Earliest_Start=("Starts", "min"),
        Latest_End=("Ends", "max"),
        Minimum_SOA=("SOA", "min"),
        Maximum_SOA=("SOA", "max")
    )
    .sort_values(
        ["SOA_Periods", "Product_ID"],
        ascending=[False, True]
    )
)

print(
    "Products with one SOA period:",
    (soa_periods_per_product["SOA_Periods"] == 1).sum()
)

print(
    "Products with multiple SOA periods:",
    (soa_periods_per_product["SOA_Periods"] > 1).sum()
)

print(
    "Maximum SOA periods for one product:",
    soa_periods_per_product["SOA_Periods"].max()
)

display(
    soa_periods_per_product.head(20)
)

Products with one SOA period: 412
Products with multiple SOA periods: 0
Maximum SOA periods for one product: 1


,Product_ID,Model,SOA_Periods,Earliest_Start,Latest_End,Minimum_SOA,Maximum_SOA
0,11,107833-01,1,2025-12-31,2026-02-03,66.50,66.50
1,36,161818-01,1,2025-12-31,2026-02-03,66.50,66.50
2,39,19750,1,2026-02-01,2026-02-28,5.00,5.00
3,48,21270,1,2026-02-01,2026-02-28,2.29,2.29
4,49,21271,1,2026-02-01,2026-02-28,2.29,2.29
5,50,21274,1,2026-02-01,2026-02-28,2.29,2.29
6,52,21640,1,2026-02-01,2026-02-28,2.29,2.29
7,53,21641,1,2026-02-01,2026-02-28,2.29,2.29
8,54,21644,1,2026-02-01,2026-02-28,2.29,2.29
9,57,23211,1,2026-02-01,2026-02-28,17.08,17.08


In [36]:
# ============================================================
#  PRODUCTS WITH MULTIPLE SOA WINDOWS
# ============================================================

multi_period_ids = soa_periods_per_product.loc[
    soa_periods_per_product["SOA_Periods"] > 1,
    "Product_ID"
]

multi_period_soa = (
    soa_work[
        soa_work["Product_ID"].isin(multi_period_ids)
    ]
    [
        [
            "Product_ID",
            "Model",
            "Description",
            "Starts",
            "Ends",
            "Window_Days",
            "SOA"
        ]
    ]
    .sort_values(
        ["Product_ID", "Starts", "Ends"]
    )
)

print(
    "SOA records belonging to multi-period products:",
    len(multi_period_soa)
)

display(multi_period_soa)

SOA records belonging to multi-period products: 0


,Product_ID,Model,Description,Starts,Ends,Window_Days,SOA


In [37]:
# ============================================================
# DETECT OVERLAPPING SOA WINDOWS
# ============================================================

soa_sorted = soa_work.sort_values(
    ["Product_ID", "Starts", "Ends"]
).copy()

soa_sorted["Previous_End"] = (
    soa_sorted
    .groupby("Product_ID")["Ends"]
    .shift(1)
)

soa_sorted["Overlap_Flag"] = (
    soa_sorted["Previous_End"].notna()
    &
    (soa_sorted["Starts"] <= soa_sorted["Previous_End"])
)

soa_overlaps = soa_sorted.loc[
    soa_sorted["Overlap_Flag"],
    [
        "Product_ID",
        "Model",
        "Description",
        "Starts",
        "Ends",
        "Previous_End",
        "SOA"
    ]
]

print(
    "SOA records with overlapping previous windows:",
    len(soa_overlaps)
)

print(
    "Products affected by overlapping windows:",
    soa_overlaps["Product_ID"].nunique()
)

display(soa_overlaps)

SOA records with overlapping previous windows: 0
Products affected by overlapping windows: 0


,Product_ID,Model,Description,Starts,Ends,Previous_End,SOA


In [38]:
## Final SOA structural validation summary
soa_validation_summary = pd.Series({
    "SOA_Rows":
        len(soa_work),

    "Mapped_Product_ID":
        soa_work["Product_ID"].notna().sum(),

    "Unmapped_Product_ID":
        soa_work["Product_ID"].isna().sum(),

    "Unique_Products":
        soa_work["Product_ID"].nunique(),

    "Duplicate_Grain_Rows":
        grain_duplicate_mask.sum(),

    "Products_With_Multiple_Periods":
        (soa_periods_per_product["SOA_Periods"] > 1).sum(),

    "Overlapping_SOA_Records":
        len(soa_overlaps),

    "Products_With_Overlaps":
        soa_overlaps["Product_ID"].nunique(),

    "Invalid_Date_Windows":
        (soa_work["Starts"] > soa_work["Ends"]).sum(),

    "Date_Corrections":
        soa_work["Date_Correction_Flag"].sum()
}).to_frame("Value")

display(soa_validation_summary)

,Value
SOA_Rows,412
Mapped_Product_ID,412
Unmapped_Product_ID,0
Unique_Products,412
Duplicate_Grain_Rows,0
Products_With_Multiple_Periods,0
Overlapping_SOA_Records,0
Products_With_Overlaps,0
Invalid_Date_Windows,0
Date_Corrections,1


In [39]:
# ============================================================
#  BUILD GOVERNED FACT_SOA
# ============================================================

fact_soa = soa_work.copy()

# Sort deterministically
fact_soa = (
    fact_soa
    .sort_values(
        ["Product_ID", "Starts", "Ends"]
    )
    .reset_index(drop=True)
)

# Surrogate fact key
fact_soa.insert(
    0,
    "SOA_Record_ID",
    range(1, len(fact_soa) + 1)
)

# Select governed columns
fact_soa = fact_soa[
    [
        "SOA_Record_ID",
        "Product_ID",
        "Product_Key",
        "Model",
        "Description",
        "Starts",
        "Ends",
        "Window_Days",
        "SOA",
        "Original_Ends",
        "Date_Correction_Flag"
    ]
].copy()

print("fact_soa shape:", fact_soa.shape)

display(fact_soa.head(10))

fact_soa shape: (412, 11)


,SOA_Record_ID,Product_ID,Product_Key,Model,Description,Starts,Ends,Window_Days,SOA,Original_Ends,Date_Correction_Flag
0,1,11,107833-01,107833-01,Dyson Supersonic Hair Dryer,2025-12-31,2026-02-03,35,66.50,2026-02-03,False
1,2,36,161818-01,161818-01,Dyson Supersonic Ceramic,2025-12-31,2026-02-03,35,66.50,2026-02-03,False
2,3,39,19750,19750,Russell Hobbs Rice Cooker 1.8Ltr,2026-02-01,2026-02-28,28,5.00,2026-02-28,False
3,4,48,21270,21270,Russell Hobbs White Textures Jug,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
4,5,49,21271,21271,Russell Hobbs Black Textures Jug,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
5,6,50,21274,21274,Russell Hobbs Texture Kettle,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
6,7,52,21640,21640,Russell Hobbs White Textures 2,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
7,8,53,21641,21641,RUSSELL HOBBS BLACK,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
8,9,54,21644,21644,Russell Hobbs Textures Toaster,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
9,10,57,23211,23211,Luna Moonlight Grey Quiet Boil,2026-02-01,2026-02-28,28,17.08,2026-02-28,False


In [40]:
# ============================================================
#  FINAL FACT_SOA VALIDATION
# ============================================================

print("Total SOA records      :", len(fact_soa))
print("Unique SOA_Record_ID   :", fact_soa["SOA_Record_ID"].nunique())
print("Unique Product_ID      :", fact_soa["Product_ID"].nunique())

print(
    "Missing Product_ID     :",
    fact_soa["Product_ID"].isna().sum()
)

print(
    "Missing SOA values     :",
    fact_soa["SOA"].isna().sum()
)

print(
    "Invalid date windows   :",
    (fact_soa["Starts"] > fact_soa["Ends"]).sum()
)

print(
    "Duplicate SOA grain    :",
    fact_soa.duplicated(
        subset=["Product_ID", "Starts", "Ends"]
    ).sum()
)

print(
    "Date corrections       :",
    fact_soa["Date_Correction_Flag"].sum()
)

Total SOA records      : 412
Unique SOA_Record_ID   : 412
Unique Product_ID      : 412
Missing Product_ID     : 0
Missing SOA values     : 0
Invalid date windows   : 0
Duplicate SOA grain    : 0
Date corrections       : 1


In [41]:
# ============================================================
# SAVE FACT_SOA
# ============================================================

""" PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FACT_SOA_FILE = (
    PROCESSED_DIR / "fact_soa.csv"
)

fact_soa.to_csv(
    FACT_SOA_FILE,
    index=False
)

print("Saved:", FACT_SOA_FILE) """

Saved: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed\fact_soa.csv


In [43]:
# ============================================================
#  RELOAD & VALIDATE SAVED FACT_SOA
# ============================================================

fact_soa_check = pd.read_csv(
    FACT_SOA_FILE,
    parse_dates=[
        "Starts",
        "Ends",
        "Original_Ends"
    ]
)

print("Saved rows          :", len(fact_soa_check))
print(
    "Unique SOA_Record_ID:",
    fact_soa_check["SOA_Record_ID"].nunique()
)

print(
    "Missing Product_ID  :",
    fact_soa_check["Product_ID"].isna().sum()
)

print(
    "Missing SOA         :",
    fact_soa_check["SOA"].isna().sum()
)

print(
    "Invalid windows     :",
    (
        fact_soa_check["Starts"]
        > fact_soa_check["Ends"]
    ).sum()
)

print(
    "Duplicate grain     :",
    fact_soa_check.duplicated(
        subset=[
            "Product_ID",
            "Starts",
            "Ends"
        ]
    ).sum()
)

print(
    "Corrections retained:",
    fact_soa_check["Date_Correction_Flag"].sum()
)

Saved rows          : 412
Unique SOA_Record_ID: 412
Missing Product_ID  : 0
Missing SOA         : 0
Invalid windows     : 0
Duplicate grain     : 0
Corrections retained: 1


In [44]:
# ============================================================
# FACT_SOA GATE
# ============================================================

fact_soa_gate = (
    len(fact_soa_check) == 412
    and fact_soa_check["SOA_Record_ID"].is_unique
    and fact_soa_check["Product_ID"].notna().all()
    and fact_soa_check["SOA"].notna().all()
    and (
        fact_soa_check["Starts"]
        <= fact_soa_check["Ends"]
    ).all()
    and fact_soa_check.duplicated(
        subset=[
            "Product_ID",
            "Starts",
            "Ends"
        ]
    ).sum() == 0
    and fact_soa_check[
        "Date_Correction_Flag"
    ].sum() == 1
)

print(
    "FACT_SOA GATE:",
    "PASSED" if fact_soa_gate else "FAILED"
)

FACT_SOA GATE: PASSED
